In [1]:


import json
from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
from tqdm import tqdm

# === File Paths ===
input_path = r"C:\Users\pauli\Downloads\EXIST2025_filtered_5yes.json"
output_path = r"C:\Users\pauli\Downloads\EXIST2025_filtered_5yes_first20_MBART-50_translated.json"

# === Load NLLB Model and Tokenizer ===
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = NllbTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# === Language Codes ===
lang_en = "eng_Latn"
lang_es = "spa_Latn"

# === Translation Function ===
def translate(text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    # Remove max_length to avoid input truncation OR increase it to model max
    encoded = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=None)
    tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    
    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tgt_lang_id,
        max_new_tokens=None,
        early_stopping=True,
        no_repeat_ngram_size=3,
        do_sample=False  # Try True if you want more diverse output
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

# === Load Filtered Entries ===
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# === Only Process First 20 Entries ===
first_20_items = list(data.items())[:20]
translated_data = {}

for key, entry in tqdm(first_20_items, desc="Translating first 20 tweets"):
    try:
        tweet = entry.get("tweet", "")
        translated_es = translate(tweet, lang_en, lang_es)
        translated_back_en = translate(translated_es, lang_es, lang_en)

        translated_data[key] = {
            "tweet": tweet,
            "translated_es": translated_es,
            "translated_back_en": translated_back_en
        }

    except Exception as e:
        print(f"⚠️ Error translating {key}: {e}")
        translated_data[key] = {
            "tweet": tweet,
            "translated_es": "",
            "translated_back_en": ""
        }

# === Save Translations ===
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(translated_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ First 20 tweets translated. Output saved to:\n{output_path}")


c:\Users\pauli\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Translating first 20 tweets:   0%|          | 0/20 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Translating first 20 tweets:   5%|▌         | 1/20 [00:16<05:14, 16.56s/it]The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Translating first 20 tweets:  10%|█         | 2/20 [00:27<03:


✅ First 20 tweets translated. Output saved to:
C:\Users\pauli\Downloads\EXIST2025_filtered_5yes_first20_MBART-50_translated.json
